# 22DM015 Final Project — Financial PhraseBank
## Person B: Parts 2 & 3 — BERT track (BERT, augmentation eval, full-data curve)

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).‍
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).‍
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).‍
- Evaluate headline numbers on **`test`** only; tune on **`val`**.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical data for everyone
train, val, test = splits['train'], splits['val'], splits['test']
labeled_32 = splits['labeled_32']
pool = du.unlabeled_pool(splits)     # Part 2 'unlabelled' data
PERSON = 'B'
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

> **Install (run once):** `transformers`, `torch`, `accelerate` are needed here.‍ On Python 3.14 torch wheels may be missing — use a 3.11/3.12 venv.‍

## Part 2a.‍ BERT with 32 labelled examples (0.5)
Fine-tune a BERT-family model on `labeled_32`, evaluate on `test`.‍ Expect instability with 32 examples — fix seed, report val + test.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Skeleton — fill in tokenizer/model/Trainer.
MODEL = 'bert-base-uncased'   # or 'ProsusAI/finbert' for the SOA comparison
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# tok = AutoTokenizer.from_pretrained(MODEL)
# ... tokenize labeled_32 / val / test, train, predict ...
# y_pred = ...
# m = eu.evaluate(test['label'], y_pred)
# eu.log_result(MODEL, '32-shot', len(labeled_32), m, person=PERSON)

## Part 2b.‍ Train on Person D's augmented set (1)
Person D produces a non-LLM augmented training set (back-translation / EDA / etc.) under `data/augmented_32.csv`.‍ Re-train the SAME BERT on it and compare to 2a.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# aug = pd.read_csv('../data/augmented_32.csv')   # produced by Person D
# train BERT on aug, evaluate on test
# eu.log_result(MODEL, 'augmented', len(aug), m, person=PERSON)

## Part 2d.‍ Train on 32 + Person D's LLM-generated data (1)
Person D produces `data/llm_generated.csv`.‍ Train BERT on the 32 + generated points; analyse impact on metrics.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# gen = pd.read_csv('../data/llm_generated.csv')
# combine labeled_32 + gen, train, evaluate
# eu.log_result(MODEL, 'llm-generated', 32+len(gen), m, person=PERSON)

## Part 2e.‍ Optimal technique (0.5)
Apply the best technique(s) from 2a/2b/2d.‍ Comment + propose improvements (student-authored).‍

## Part 3.‍ Full-dataset SOA comparison (2)
3a.‍ Train on 1/10/25/50/75/100% of `train` (use `du.subset_by_fraction`).‍ 3b.‍ Plot the learning curve.‍ 3c.‍ Fold in Part 2 techniques.‍ 3d.‍ Methodology analysis.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
FRACTIONS = [0.01, 0.10, 0.25, 0.50, 0.75, 1.00]
for f in FRACTIONS:
    sub = du.subset_by_fraction(train, f)
    print(f'frac={f:>4}: n={len(sub)}', dict(sub['label_name'].value_counts()))
    # train BERT on `sub`, predict on test
    # m = eu.evaluate(test['label'], y_pred)
    # eu.log_result(MODEL, f'full-{int(f*100)}%', len(sub), m, person=PERSON)

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# 3b. Learning curve — read everyone's rows back and plot.
res = pd.read_csv('../results/results.csv')
print(res)
# import matplotlib.pyplot as plt; plot f1_macro vs n_train_labeled